In [0]:
from pyspark.sql.functions import lit , col , when , current_timestamp , max , count , sum 

In [0]:
bronze_table = "globalretail.bronze.sales_raw"

silver_table = "globalretail.silver.fact_sales"

hwm_table = "globalretail.validation.silver_watermarks"

hwm_last_loaded_at = spark.sql(f"SELECT last_successful_load_at from {hwm_table} where entity_name='transaction'").collect()[0][0]
print(hwm_last_loaded_at)


max_last_loaded_at = spark.sql(f" SELECT max(ingested_at) from {bronze_table} where ingested_at > '{hwm_last_loaded_at}'").collect()[0][0]

print(max_last_loaded_at)


bronze_df = spark.read.table(bronze_table)

new_row_df = bronze_df.filter(col("ingested_at") > hwm_last_loaded_at)

display(new_row_df)

In [0]:
new_row_df.createOrReplaceTempView("vw_transaction_from_bronze")

In [0]:
cleaned = spark.sql("""
SELECT
    *,

    CASE
        WHEN UPPER(TRIM(channel)) IN ('WEB', 'ONLINE', 'ECOM', 'ECOMMERCE', 'APP') THEN 'ONLINE'
        WHEN UPPER(TRIM(channel)) IN ('STORE', 'POS', 'RETAIL') THEN 'STORE'
        WHEN UPPER(TRIM(channel)) = 'MARKETPLACE' THEN 'MARKETPLACE'
        ELSE UPPER(TRIM(channel))
    END AS channel_std,

    UPPER(TRIM(currency_code)) AS currency_code_std,

    ROUND(quantity * unit_price, 2) AS gross_amount,

    ROUND((quantity * unit_price) - discount_amount + tax_amount, 2) AS net_amount,

    CASE
        WHEN product_id IS NULL THEN 'MISSING_PRODUCT_ID'
        WHEN quantity IS NULL OR quantity <= 0 THEN 'INVALID_QUANTITY'
        WHEN UPPER(TRIM(currency_code)) NOT IN ('USD', 'EUR', 'GBP', 'INR') THEN 'INVALID_CURRENCY'
        ELSE NULL
    END AS reject_reason

FROM vw_transaction_from_bronze
""")
display(cleaned)

In [0]:

rejects = cleaned.filter(col("reject_reason").isNotNull()).select("order_id", "order_line_id", "reject_reason", current_timestamp().alias("rejected_at"))
reject_count = rejects.count()
if reject_count > 0:
    rejects.write.mode("append").saveAsTable("globalretail.silver.sales_reject")

In [0]:
valid_trx = cleaned.filter(col("reject_reason").isNull())
valid_trx.createOrReplaceTempView("vw_valid_trx")

In [0]:
fact_df = spark.sql(f"""
SELECT
    SHA2(CONCAT(t.order_id, '||', CAST(t.order_line_id AS STRING)), 256) AS sales_sk,
    t.order_id,
    t.order_line_id,
    CAST(t.order_ts_utc AS TIMESTAMP) AS order_ts_utc,
    CAST(date_format(to_date(t.order_ts_utc), 'yyyyMMdd') AS INT) AS date_key,
    CAST(COALESCE(dc.customer_sk, -1) AS BIGINT) AS customer_sk,
    CAST(dp.product_sk AS BIGINT) AS product_sk,
    CAST(ds.store_sk AS BIGINT) AS store_sk,
    CAST(t.customer_id AS BIGINT) AS customer_id,
    CAST(t.product_id AS BIGINT) AS product_id,
    CAST(t.store_id AS BIGINT) AS store_id,
    t.channel_std AS channel,
    UPPER(TRIM(t.payment_status)) AS payment_status,
    CAST(t.quantity AS INT) AS quantity,
    CAST(t.unit_price AS DECIMAL(18,2)) AS unit_price,
    CAST(t.discount_amount AS DECIMAL(18,2)) AS discount_amount,
    CAST(t.tax_amount AS DECIMAL(18,2)) AS tax_amount,
    CAST(t.gross_amount AS DECIMAL(18,2)) AS gross_amount,
    CAST(t.net_amount AS DECIMAL(18,2)) AS net_amount,
    CAST(
        ROUND(
            t.net_amount * CASE
                WHEN t.currency_code_std = 'USD' THEN 1.0
                WHEN t.currency_code_std = 'EUR' THEN 1.09
                WHEN t.currency_code_std = 'GBP' THEN 1.27
                WHEN t.currency_code_std = 'INR' THEN 0.012
                ELSE NULL
            END,
            2
        ) AS DECIMAL(18,2)
    ) AS usd_net_amount,
    CAST(
        ROUND(t.net_amount - (t.quantity * dp.unit_cost), 2)
        AS DECIMAL(18,2)
    ) AS gross_margin_amount,
    t.currency_code_std AS currency_code,
    CAST(
        CASE
            WHEN t.currency_code_std = 'USD' THEN 1.0
            WHEN t.currency_code_std = 'EUR' THEN 1.09
            WHEN t.currency_code_std = 'GBP' THEN 1.27
            WHEN t.currency_code_std = 'INR' THEN 0.012
            ELSE NULL
        END AS DECIMAL(18,6)
    ) AS fx_rate_to_usd,
    CAST(t.updated_at AS TIMESTAMP) AS updated_at
FROM vw_valid_trx t
LEFT JOIN globalretail.silver.dim_customers dc
    ON t.customer_id = dc.customer_id
   AND CAST(t.order_ts_utc AS TIMESTAMP) >= dc.effective_start_ts
   AND CAST(t.order_ts_utc AS TIMESTAMP) <= dc.effective_end_ts
LEFT JOIN globalretail.silver.dim_products dp
    ON t.product_id = dp.product_id
   AND CAST(t.order_ts_utc AS TIMESTAMP) >= dp.effective_start_ts
   AND CAST(t.order_ts_utc AS TIMESTAMP) <= dp.effective_end_ts
LEFT JOIN globalretail.silver.dim_store ds
    ON t.store_id = ds.store_id
""")
display(fact_df)

In [0]:
fact_df.createOrReplaceTempView("vw_fact_sales")

spark.sql(f"""
MERGE INTO globalretail.silver.fact_sales t
USING vw_fact_sales s
ON t.sales_sk = s.sales_sk
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


In [0]:

if max_last_loaded_at is not None:
    spark.sql(f"""
    UPDATE globalretail.validation.silver_watermarks
    SET   last_successful_load_at =  TIMESTAMP('{max_last_loaded_at}')
    WHERE entity_name = 'transaction'
    """)
else:
    print("No new records to process — watermark not updated.")